# DiffShield job runner
Set `JOB_IDS` in the first code cell, then **Run All** (or *Save Version -> Save & Run All*).
Nothing else needs editing. Jobs (sessions) are defined in `kaggle/jobs.yaml` inside the code bundle; the launch table
is in `docs/KAGGLE_LAUNCH.md`.

Checklist before running: Internet **ON**; accelerator **GPU T4 x2** (J00 / S03: none is enough); Add Data: the code
dataset (`diffshield-code`), the CelebAMask-HQ dataset (`ipythonx/celebamaskhq`) for every job that needs images, and the
output of an earlier session when the table says so (then set `PARAMS = {"prev_root": "/kaggle/input/<its name>"}`).
No tokens or logins are needed or should be entered.

In [ ]:
# ---- the only cell you edit -------------------------------------------------------------
JOB_IDS      = ["S04"]      # run in this order, e.g. ["S06"] or ["S10", "S12"]
PARAMS       = {}           # e.g. {"smoke": True}  (5 images, 5 iterations, 1 edit seed)  or  {"prev_root": "/kaggle/input/s06-output"}
RESUME_FROM  = None         # path of THIS session's earlier output added as an input, e.g. "/kaggle/input/s06-output-run1"
JOB_SHARD    = None         # e.g. "0/2": process only half of every job's items (the other half on another account)
FORCE        = False        # True: rerun jobs that already finished in this session
# -------------------------------------------------------------------------------------------

In [ ]:
import os, sys, json, shutil, zipfile, subprocess
from pathlib import Path

print("inputs attached to this notebook:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print("  ", p)

# the code is unpacked OUTSIDE /kaggle/working so that it is not copied into this notebook's output
CODE_DIR = Path("/kaggle/tmp/DiffShield") if Path("/kaggle/tmp").exists() or os.access("/kaggle", os.W_OK) else Path("/kaggle/working/_code")
CODE_DIR.parent.mkdir(parents=True, exist_ok=True)

def find_bundles(root="/kaggle/input"):
    found = []
    for r, dirs, files in os.walk(root):
        if "BUNDLE_INFO.json" in files:
            found.append(("dir", Path(r)))
            dirs[:] = []
            continue
        for f in files:
            if f.startswith("diffshield-code") and f.endswith(".zip"):
                found.append(("zip", Path(r) / f))
        if r[len(root):].count(os.sep) >= 3:
            dirs[:] = []
    return found

def created(item):
    kind, p = item
    try:
        if kind == "dir":
            return json.loads((p / "BUNDLE_INFO.json").read_text()).get("created_utc", "")
        with zipfile.ZipFile(p) as z:
            return json.loads(z.read("BUNDLE_INFO.json")).get("created_utc", "")
    except Exception:
        return ""

cands = find_bundles()
pref = [c for c in cands if "diffshield-code" in str(c[1])] or cands
assert pref, "code bundle not found under /kaggle/input: add the diffshield-code dataset to this notebook"
kind, p = max(pref, key=created)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
if kind == "dir":
    shutil.copytree(p, CODE_DIR)
else:
    with zipfile.ZipFile(p) as z:
        z.extractall(CODE_DIR)
print("code bundle:", p, "->", CODE_DIR)
print(json.dumps({k: v for k, v in json.loads((CODE_DIR / "BUNDLE_INFO.json").read_text()).items() if k != "sha256"}, indent=2))

In [ ]:
cmd = [sys.executable, "scripts/run_job.py", "--jobs", ",".join(JOB_IDS), "--work", "/kaggle/working"]
if JOB_SHARD: cmd += ["--job_shard", JOB_SHARD]
if RESUME_FROM: cmd += ["--resume_from", RESUME_FROM]
if FORCE: cmd += ["--force"]
for k, v in PARAMS.items(): cmd += ["--set", f"{k}={json.dumps(v)}"]
print(" ".join(cmd))
env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
p = subprocess.Popen(cmd, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print("run_job exit code:", p.returncode)

In [ ]:
# Summary of what was produced (download the *_outputs.zip files from the Output tab afterwards)
for f in sorted(Path("/kaggle/working").glob("*_outputs.zip")):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")
sm = Path("/kaggle/working/job_runner_summary.json")
if sm.exists():
    for k, v in json.loads(sm.read_text()).items():
        print(k, "ok" if v.get("ok") else "FAILED " + str(v.get("failed_steps")), round(v["session_seconds"]), "s")
assert p.returncode == 0, "a job failed: see /kaggle/working/<JOB>/logs/ ; rerun the same cell to resume"